# Static XAS shutter check

Inspect the shutter sectioning used for a processed static-XAS file. The main plot follows the first shutter diagnostic plot in `legacy/xas_process_scan_single_run.ipynb`.

In [ ]:
import sys
from pathlib import Path

import h5py
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

cwd = Path.cwd().resolve()
repo_root = None
for p in [cwd] + list(cwd.parents):
    if (p / "analysis" / "scripts").exists():
        repo_root = p
        break
if repo_root is None:
    raise RuntimeError("Could not locate repository root containing analysis/scripts")

scripts_dir = repo_root / "analysis" / "scripts"
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

import config
import data_loading
import compute_static_xas as csx


## Select processed file

In [ ]:
# Set H5_PATH explicitly if you do not want the newest file found below.
H5_PATH = None

candidate_dirs = [
    repo_root / "processed" / "static_xas",
    repo_root / "processed" / "xas_static",
    repo_root / "11022188" / "processed" / "static_xas",
    repo_root / "11022188" / "processed" / "xas_static",
    Path(config.COMBINED_DIR).parent / "static_xas",
    Path(config.COMBINED_DIR).parent / "xas_static",
]

if H5_PATH is None:
    found = []
    for d in candidate_dirs:
        if d.exists():
            found.extend(d.glob("*.h5"))
    if not found:
        print("Searched:")
        for d in candidate_dirs:
            print("  ", d)
        raise FileNotFoundError("No processed static-XAS H5 files found. Set H5_PATH manually.")
    H5_PATH = max(found, key=lambda p: p.stat().st_mtime)
else:
    H5_PATH = Path(H5_PATH)

print("Using:", H5_PATH)


## Processed-file summary

In [ ]:
with h5py.File(H5_PATH, "r") as f:
    energies = f["nominal_energies"][:]
    n_shots = f["n_shots"][:]
    vls_shape = f["vls"].shape
    gmd_shape = f["gmd"].shape
    attrs = {k: f.attrs[k] for k in f.attrs.keys()}

print("mode:", attrs.get("mode"))
print("run_no:", attrs.get("run_no"))
print("config_path:", attrs.get("config_path", "<none>"))
print("n_sections_detected:", attrs.get("n_sections_detected"))
print("n_sections_used:", attrs.get("n_sections_used"))
print("vls shape:", vls_shape)
print("gmd shape:", gmd_shape)
print("total shots:", int(np.nansum(n_shots)))

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.bar(np.arange(len(energies)), n_shots, color="tab:green", alpha=0.75)
ax.set_xticks(np.arange(len(energies)))
ax.set_xticklabels([f"{e:.1f}" for e in energies], rotation=90)
ax.set_xlabel("Nominal photon energy (eV)")
ax.set_ylabel("Kept shots")
ax.set_title("Processed static-XAS shot count per section")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
plt.show()


## Recompute shutter sections from raw data

In [ ]:
def _as_tuple2(x):
    return tuple(int(v) for v in np.asarray(x).ravel()[:2])

cfg = None
config_path_attr = attrs.get("config_path", None)
if config_path_attr is not None:
    config_path = Path(str(config_path_attr))
    if not config_path.is_absolute():
        config_path = repo_root / config_path
    if config_path.exists():
        cfg = csx._load_config(config_path)
        print("Loaded config:", config_path)

run_no = attrs["run_no"]
run_numbers = csx._normalize_run_numbers(run_no)
raw_dir = Path(str(attrs.get("raw_dir", getattr(cfg, "RAW_DIR", config.RAW_H5_DIR))))
max_files = getattr(cfg, "MAX_FILES", None) if cfg is not None else None
train_length = getattr(cfg, "TRAIN_LENGTH", None) if cfg is not None else None
crop_roi = _as_tuple2(attrs["vls_crop_roi"])
signal_bunch_range = _as_tuple2(attrs["signal_bunch_range"])
bg_bunch_range = _as_tuple2(attrs["bg_bunch_range"])
vls_bunch_roll = int(attrs.get("vls_bunch_roll", 0))
train_rate_hz = float(attrs.get("train_rate_hz", 10.0))
transition_trim_seconds = float(attrs.get("transition_trim_seconds", 3.0))
shutter_index_path = str(attrs.get("shutter_index_path", csx._DEFAULT_SHUTTER_INDEX_PATH))
shutter_value_path = str(attrs.get("shutter_value_path", csx._DEFAULT_SHUTTER_VALUE_PATH))

print("runs:", run_numbers)
print("raw_dir:", raw_dir)
print("max_files:", max_files)
print("crop_roi:", crop_roi)
print("signal_bunch_range:", signal_bunch_range)
print("bg_bunch_range:", bg_bunch_range)

data = csx._concat_experiment_data([
    data_loading.load_raw_h5(
        run, config=2, raw_dir=raw_dir,
        train_length=train_length, max_files=max_files,
    )
    for run in run_numbers
])
data = data.crop_vls(*crop_roi)
data = data.roll_vls_bunches(vls_bunch_roll)
data = data.auto_subtract_background_trainwise(bg_bunch_range)

sig_b0, sig_b1 = signal_bunch_range
vls = np.asarray(data.vls, dtype=np.float64)
train_score = np.nanmean(np.nansum(vls[:, sig_b0:sig_b1, :], axis=2), axis=1)

h5_paths = csx._list_raw_h5_files_for_runs(run_numbers, raw_dir, max_files=max_files)
shutter = csx._read_aligned_shutter(h5_paths, shutter_index_path, shutter_value_path, data.tID)
trim_n = int(round(transition_trim_seconds * train_rate_hz))
open_blocks, closed_blocks, is_open_raw, is_closed_raw, valid_mask = csx._detect_sections(
    shutter, train_score, trim_n
)

n_used = min(len(open_blocks), len(energies))
open_blocks_used = open_blocks[:n_used]
section_idx = np.full(is_open_raw.shape, -1, dtype=np.int32)
for i, (s, e) in enumerate(open_blocks_used):
    section_idx[s:e] = i
is_open = (section_idx >= 0) & valid_mask & is_open_raw
is_closed = is_closed_raw & valid_mask

print(f"Detected open sections: {len(open_blocks)}")
print(f"Detected closed sections: {len(closed_blocks)}")
print(f"Sections used in processed file: {n_used}")
print(f"Transition trim: +- {transition_trim_seconds:.1f} s ({trim_n} trains)")
print(f"Open trains kept: {int(np.sum(is_open))}")
print(f"Closed trains kept: {int(np.sum(is_closed))}")
print(f"Mean train score (open kept): {np.nanmean(train_score[is_open]):.3g}")
print(f"Mean train score (closed kept): {np.nanmean(train_score[is_closed]):.3g}")


## Shutter diagnostic plot

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.8))
ax.plot(train_score, lw=0.8, color="0.5", label="train score")
ax.scatter(np.where(is_open_raw)[0], train_score[is_open_raw], s=5, color="lightcoral", label="open raw")
ax.scatter(np.where(is_open)[0], train_score[is_open], s=10, color="tab:red", label="open kept")
ax.scatter(np.where(is_closed)[0], train_score[is_closed], s=10, color="tab:blue", label="closed kept")

y_text = np.nanmax(train_score)
for i, (s, e) in enumerate(open_blocks_used):
    ax.axvline(s, ls="--", lw=0.8, color="k", alpha=0.5)
    ax.axvline(e, ls="--", lw=0.8, color="k", alpha=0.5)
    ax.text(s, y_text, f"{energies[i]:.1f} eV", fontsize=7, va="bottom")

ax.set_xlabel("Train index after concatenating selected run(s)")
ax.set_ylabel("VLS score (arb.)")
ax.set_title(f"Fast-shutter sections for {H5_PATH.name}")
ax.legend(loc="best")
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()


## Section table

In [ ]:
print(f"{'idx':>3s}  {'E_nom/eV':>8s}  {'open block':>17s}  {'kept trains':>11s}  {'shots H5':>9s}")
print("-" * 60)
for i, (s, e) in enumerate(open_blocks_used):
    kept = int(np.sum((section_idx == i) & valid_mask & is_open_raw))
    shots = int(n_shots[i]) if i < len(n_shots) else 0
    print(f"{i:3d}  {energies[i]:8.2f}  [{s:6d}, {e:6d})  {kept:11d}  {shots:9d}")
